# Moon Rover MPC Testing

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import functools
import jax.numpy as jnp
import numpy as np
import tqdm
import pandas as pd

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.quartic_cost as quartic_cost

In [ ]:
def load_specific_sms_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    df = pd.read_csv(file_path)

    ks = df.keys()

    ts = np.array(df[ks[0]])
    diff = np.abs(np.diff(ts))
    avg_diff = np.mean(diff)
    std_diff = np.std(diff)
    if std_diff > 0.05:
        bad_indices = np.where(diff > avg_diff + std_diff)[0] + 1 # off by one
        start_index = bad_indices[-2] + 5 * 200
        end_index = bad_indices[-1] - 1
    else:
        start_index = 0
        end_index = ts.size - 1

    df = df[start_index: end_index + 1]

    acc_ref = jnp.transpose(jnp.array([df[k] for k in ks[1:4]]))
    omega_ref = jnp.transpose(jnp.array([df[k] for k in ks[4:]]))
    return acc_ref, omega_ref

In [ ]:
# load data

# file_path = "/Users/jozbee/work/eng/comp/data/sms_00_sms_drive.csv"
file_path = "/Users/jozbee/work/eng/comp/data/sms_specific-forces-standard-road-v2.csv"
acc_ref, omega_ref = load_specific_sms_references(file_path)

assert acc_ref.shape[0] == omega_ref.shape[0]
assert acc_ref.shape[1] == 3
assert omega_ref.shape[1] == 3

In [ ]:
# general setup
begin = 0
num_steps = 100 * 200
# num_steps = acc_ref.shape[0]
n = 200  # horizon

acc_ref = acc_ref[:num_steps]
omega_ref = omega_ref[:num_steps]

In [ ]:
# cost setup
margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
euler_margins = [0.2 / 3.0, 0.1 / 3.0]
euler_sizes = [2**0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    # margins=[0.6],
    # sizes=[1.0, 25],
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    # margins=euler_margins,
    # sizes=euler_sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
roll_cost = quartic_cost.QuarticCost.from_bounds(
    # margins=[0.3, 0.2, 0.1],
    # sizes=[1.0, 2**3, 2**5, 2**10],
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_roll,
    high=const.max_roll,
)
pitch_cost = quartic_cost.QuarticCost.from_bounds(
    # margins=[0.3, 0.2, 0.1],
    # sizes=[1.0, 2**3, 2**5, 2**10],
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_pitch,
    high=const.max_pitch,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_yaw,
    high=const.max_rotary_yaw,
)
yaw_dot_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_vel,
    high=const.max_rotary_vel,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    roll_cost=roll_cost,
    pitch_cost=pitch_cost,
    yaw_cost=yaw_cost,
    yaw_dot_cost=yaw_dot_cost,
)

In [ ]:
# run setup

weights0 = opt.ExpWeights(
    acc=jnp.array([1e2, 1e2, 1e0]),
    omega=jnp.array([2e2, 2e2, 2e2]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

# def state_err(state: opt.TrainState) -> jax.Array:
#     return jnp.sum(jnp.square(state.vstate0_irl - state.vstate0_sim))

def run_sim(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    w: jax.Array,
) -> jax.Array:
    assert acc_ref.shape[0] == omega_ref.shape[0]
    assert acc_ref.shape[1] == 3 and omega_ref.shape[1] == 3
    assert w.size == 4

    weights = opt.ExpWeights(
        acc=jnp.array([w[0], w[0], 1e0]),
        omega=jnp.ones(3) * w[1],
        alpha_acc=jnp.array([w[2]]),
        alpha_omega=jnp.array(w[3]),
    )
    train_step = functools.partial(opt.train_step_with_cost_jax, weights, cost_terms)
    train_state0 = opt.TrainState.zero_init(n, ("earth", "moon"))

    def train_loop(train_state, refs):
        aref, oref = refs
        assert aref.shape == (3,)
        assert oref.shape == (3,)
        aref = jnp.tile(aref, reps=(n, 1))
        oref = jnp.tile(oref, reps=(n, 1))
        new_state, _, _ = train_step(
            aref, oref, train_state, max_iter=4, max_ls=1, unroll=True
        )
        return new_state, new_state

    _, states = jax.lax.scan(train_loop, train_state0, (acc_ref, omega_ref))

    # off by one error, which I don't care to correct...
    return jnp.mean(jnp.square(states.vstate0_irl - states.vstate0_sim))


In [ ]:
# run_sim_jit = jax.jit(run_sim)
# run_sim_jit = jax.jit(jax.value_and_grad(run_sim, argnums=2))
run_sim_jit = jax.jit(jax.jacfwd(run_sim, argnums=2))

run_sim_jit(acc_ref, omega_ref, jnp.array([1e2, 2e2, 0.0, 0.0]))